# Lesson 05 - Agentic RAG


## Setup

Dis notebook show how to use Agentic RAG (Retrieval-Augmented Generation) pattern wit Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment set up through environment variables
- Azure CLI logged in (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wetin Agentic RAG be?

Traditional RAG dey follow one set way: e go find documents, then e go generate answer. **Agentic RAG** move one step further by to allow the agent make e own decision about **when** and **how** to find information.

With Agentic RAG, the agent fit:
- **Decide** whether e need to find something before e answer question
- **Choose** which data source or tool to use
- **Evaluate** the results wey e don find and fit do more retrieval if the first one no too complete
- **Combine** information from different retrieval steps to form one correct answer

Dis one make the agent dey more flexible and correct pass just one fixed retrieve-then-generate way.


## Creating a Search Tool

For Agentic RAG, external data sources na **tools** wey the agent fit use anytime e want. Dis one dey allow the agent make retrieval just be one kind action e fit do, no be say e be necessary step.

Below we go define travel knowledge base and make am available as tool weh the agent fit use to look up destination information.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Building the RAG Agent

Now we dey create agent wey dem tell make e **always find information before e answer**. The agent dey use the `search_travel_knowledge` tool to base im responses for the knowledge base instead of to depend on im own training data.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterative Retrieval — The Maker-Checker Pattern

One important advantage of Agentic RAG na **iterative retrieval**. The agent fit do plenty rounds of search to confirm, correct, or add more to the first findings — e be like "maker-checker" workflow:

1. **Maker step**: The agent go find the first information and draft one response.
2. **Checker step**: The agent go do more retrievals to confirm details or complete missing parts.

Below, dem ask the agent one question wey need am to compare many destinations, e go make am search many times.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

For dis lesson you learn how to build **Agentic RAG** system wit Microsoft Agent Framework:

- **Agentic RAG** make agents fit decide by demself when to find information, making retrieval dey dynamic no be fixed.
- **Tools as data sources**: External knowledge bases (like Azure AI Search) dem wrap as tools wey agent fit use.
- **Iterative retrieval**: The maker-checker pattern make agent fit do plenty retrieval rounds — dey search, check, and correct — before e give final answer.

For production, you go change the in-memory `TRAVEL_KNOWLEDGE_BASE` wit real Azure AI Search index to handle big travel document retrieval.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don been translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even though we try make am correct, abeg sabi say automated translations fit get errors or mistakes. Di original document for im own language na di correct source. If na serious matter, make person human professional translate am. We no go take responsibility if anybody misunderstand or misinterpret anything from dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
